# Tutorial 6: Combined Analysis

This notebook brings everything together. We show how to:

1. **Build an embedding matrix** from a list of gene perturbations → `AnnData`
2. **Build a molecule embedding matrix** from an `AnnData` whose `.obs`
   column contains drug names or SMILES
3. **Filter** failed embeddings
4. **Reduce dimensionality** with PCA (+ StandardScaler)
5. **Combine** multiple perturbation spaces (genes + drugs) into one

All of this is handled by the `PerturbationProcessor` class and the
`reduce_embeddings` convenience function.

> **Note**: The examples below use `esm2_650M` for genes and
> `chemberta2MTR` for molecules, but you can swap in any supported model.
> For molecules, `embpy` now also supports:
> - **GNN models**: `minimol` (GINE), `mhg_gnn` (graph autoencoder), `mole` (graph transformer)
> - **RDKit fingerprints**: `rdkit_fp`, `morgan_fp`, `morgan_count_fp`, `maccs_fp`, etc.
>
> See [Tutorial 4 – Molecule Embeddings](04_molecule_embeddings.ipynb) for the
> full list and installation instructions.

In [ ]:
import logging
logging.basicConfig(level=logging.WARNING)

In [ ]:
import numpy as np
import anndata as ad

from embpy.embedder import BioEmbedder
from embpy.pp.basic import PerturbationProcessor, reduce_embeddings

## 1. Build embedding matrices

`PerturbationProcessor` provides two dedicated builder methods:

| Method | Input | Use case |
|---|---|---|
| `build_embedding_matrix` | gene identifiers (list) | DNA / protein / text gene embeddings |
| `build_molecule_embedding_matrix` | drug identifiers (list **or** AnnData + column) | Transformer, GNN, or RDKit molecule embeddings |

Both return an `AnnData` with:

- `.obsm[key]` – the embedding matrix (n_perturbations × embedding_dim)
- `.obs["embedded"]` – boolean flag per perturbation (True if successful)
- `.obs["smiles"]` – resolved SMILES (molecule method only)

The `model` parameter accepts any registered model key. For molecules this
includes transformer models (`chemberta2MTR`, `chemberta2MLM`,
`molformer_base`), GNN models (`minimol`, `mhg_gnn`, `mole`), and
RDKit fingerprint types.

In [ ]:
embedder = BioEmbedder(device="auto")
pp = PerturbationProcessor(embedder=embedder)

print(f"Device: {embedder.device}")

### 1.1 Gene (protein) embedding matrix

In [ ]:
gene_list = ["TP53", "BRCA1", "EGFR", "KRAS", "MYC", "PTEN", "RB1", "AKT1"]

adata_genes = pp.build_embedding_matrix(
    identifiers=gene_list,
    model="esm2_650M",         # protein model
    id_type="symbol",
    organism="human",
    pooling_strategy="mean",
    obsm_key="X_esm2",
)

print(adata_genes)
print(f"Embedding matrix shape: {adata_genes.obsm['X_esm2'].shape}")
print(f"Successful: {adata_genes.obs['embedded'].sum()} / {len(adata_genes)}")

### 1.2 Molecule embedding matrix – from a list of drug names

The simplest way: pass a plain list of drug names (or SMILES).
`build_molecule_embedding_matrix` will automatically resolve names
to SMILES via PubChem and embed them.

In [ ]:
drug_names = ["aspirin", "ibuprofen", "caffeine", "paclitaxel", "metformin"]

adata_drugs = pp.build_molecule_embedding_matrix(
    identifiers=drug_names,
    id_type="drug_name",          # resolve names → SMILES via PubChem
    model="chemberta2MTR",
    pooling_strategy="mean",
    obsm_key="X_chemberta",
)

print(adata_drugs)
print(f"\nEmbedding matrix shape: {adata_drugs.obsm['X_chemberta'].shape}")
print(f"Successful: {adata_drugs.obs['embedded'].sum()} / {len(adata_drugs)}")
print(f"\nResolved SMILES:")
print(adata_drugs.obs[["drug_id", "smiles", "embedded"]])

In [ ]:
### 1.3  Molecule embedding matrix – from an existing AnnData

# More commonly, you already have an AnnData object (e.g. from a
# perturbation screen) whose .obs contains a column with drug identifiers.
# Just point `build_molecule_embedding_matrix` at that column.

import pandas as pd

# Simulate an existing screen AnnData with drug metadata in .obs
screen_obs = pd.DataFrame({
    "drug_name": ["aspirin", "ibuprofen", "caffeine", "paclitaxel", "metformin"],
    "dose_uM":  [10.0,       5.0,         1.0,        0.1,          50.0],
    "cell_line": ["A549"] * 5,
})
adata_screen = ad.AnnData(obs=screen_obs)
print("Input AnnData:")
print(adata_screen.obs)
print()

In [ ]:
# Build embeddings directly from the AnnData's .obs column
adata_screen_emb = pp.build_molecule_embedding_matrix(
    adata=adata_screen,
    column="drug_name",             # which .obs column has the identifiers
    id_type="drug_name",            # they are drug names, not SMILES
    model="chemberta2MTR",
    pooling_strategy="mean",
    obsm_key="X_chemberta",
)

print("Output AnnData (original metadata preserved + new columns):")
print(adata_screen_emb.obs)
print(f"\nEmbedding matrix in .obsm: {adata_screen_emb.obsm['X_chemberta'].shape}")
print(f"Original metadata columns preserved: {[c for c in ['dose_uM', 'cell_line'] if c in adata_screen_emb.obs.columns]}")

In [ ]:
# If your AnnData already has SMILES in a column, use id_type="smiles"
# to skip PubChem resolution entirely:
screen_obs_smi = pd.DataFrame({
    "smiles_col": [
        "CC(=O)Oc1ccccc1C(O)=O",  # aspirin
        "CC(C)Cc1ccc(cc1)C(C)C(O)=O",  # ibuprofen
        "Cn1c(=O)c2c(ncn2C)n(C)c1=O",  # caffeine
    ],
    "dose_uM": [10.0, 5.0, 1.0],
})
adata_smi = ad.AnnData(obs=screen_obs_smi)

adata_smi_emb = pp.build_molecule_embedding_matrix(
    adata=adata_smi,
    column="smiles_col",
    id_type="smiles",           # no name resolution needed
    model="chemberta2MTR",
    obsm_key="X_chemberta",
)

print("SMILES-column workflow:")
print(adata_smi_emb.obs[["smiles_col", "smiles", "embedded"]])
print(f"Embedding shape: {adata_smi_emb.obsm['X_chemberta'].shape}")

## 2. Filter failed embeddings

`filter_failed_embeddings` removes rows where embedding failed.

In [ ]:
adata_genes_clean = pp.filter_failed_embeddings(adata_genes, obsm_key="X_esm2")
adata_drugs_clean = pp.filter_failed_embeddings(adata_drugs, obsm_key="X_chemberta")

print(f"Genes – before: {len(adata_genes)}, after: {len(adata_genes_clean)}")
print(f"Drugs – before: {len(adata_drugs)}, after: {len(adata_drugs_clean)}")

## 3. Dimensionality reduction with PCA

`reduce_embeddings` applies PCA to the embedding matrix in `.obsm`.
It optionally applies `StandardScaler` before PCA (recommended when
combining heterogeneous embedding sources later).

Parameters:
- `obsm_key` – which `.obsm` slot to reduce
- `n_components` – PCA target dimension
- `scale` – whether to apply StandardScaler before PCA (default: `True`)
- `output_key` – name for the reduced matrix (default: `"{obsm_key}_pca"`)

In [ ]:
# Reduce gene embeddings to 50 dimensions
adata_genes_clean = reduce_embeddings(
    adata_genes_clean,
    obsm_key="X_esm2",
    n_components=50,
    scale=True,
)

print(f"Original embedding: {adata_genes_clean.obsm['X_esm2'].shape}")
print(f"Reduced embedding:  {adata_genes_clean.obsm['X_esm2_pca'].shape}")

# PCA parameters are stored in .uns
pca_info = adata_genes_clean.uns.get("X_esm2_pca_params", {})
print(f"Variance explained:  {pca_info.get('variance_explained_ratio_sum', 'N/A'):.4f}")

In [ ]:
# Same for drug embeddings
adata_drugs_clean = reduce_embeddings(
    adata_drugs_clean,
    obsm_key="X_chemberta",
    n_components=50,
    scale=True,
)

print(f"Drug original: {adata_drugs_clean.obsm['X_chemberta'].shape}")
print(f"Drug reduced:  {adata_drugs_clean.obsm['X_chemberta_pca'].shape}")

## 4. Combine perturbation spaces

After reducing both gene and drug embeddings to the same dimension
(e.g. 50), we can concatenate them into a single unified perturbation
space using `combine_perturbation_spaces`.

In [ ]:
# To combine, both AnnData objects need the same obsm key.
# Let's rename the PCA results to a common key.
adata_genes_clean.obsm["X_combined"] = adata_genes_clean.obsm["X_esm2_pca"]
adata_drugs_clean.obsm["X_combined"] = adata_drugs_clean.obsm["X_chemberta_pca"]

combined = pp.combine_perturbation_spaces(
    adata_genes_clean,
    adata_drugs_clean,
    labels=["gene", "drug"],
    obsm_key="X_combined",
)

print(combined)
print(f"\nCombined embedding shape: {combined.obsm['X_combined'].shape}")
print(f"\nPerturbation types:")
print(combined.obs["perturbation_type"].value_counts())

## 5. Visualize the combined space

We can use UMAP or PCA to visualize the combined perturbation space
in 2D.

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

X = combined.obsm["X_combined"]
pca_2d = PCA(n_components=2).fit_transform(X)

fig, ax = plt.subplots(figsize=(8, 6))

for ptype in combined.obs["perturbation_type"].unique():
    mask = combined.obs["perturbation_type"] == ptype
    ax.scatter(
        pca_2d[mask, 0], pca_2d[mask, 1],
        label=ptype, s=80, alpha=0.8, edgecolors="k", linewidth=0.5,
    )

# Label points – use "identifier" for genes, "drug_id" for drugs
label_col = "identifier" if "identifier" in combined.obs.columns else "drug_id"
for i in range(combined.n_obs):
    name = combined.obs[label_col].iloc[i] if label_col in combined.obs.columns else str(i)
    ax.annotate(name, (pca_2d[i, 0], pca_2d[i, 1]),
                fontsize=8, ha="left", va="bottom")

ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("Combined Gene + Drug Perturbation Space")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Saving and loading results

Since everything is stored in `AnnData`, you can save and reload
the entire perturbation space (embeddings + metadata) as a single
`.h5ad` file.

In [ ]:
import tempfile, os

with tempfile.TemporaryDirectory() as tmpdir:
    path = os.path.join(tmpdir, "perturbation_space.h5ad")
    combined.write_h5ad(path)
    print(f"Saved to {path} ({os.path.getsize(path):,} bytes)")

    # Reload
    loaded = ad.read_h5ad(path)
    print(f"\nReloaded: {loaded}")
    print(f"obsm keys: {list(loaded.obsm.keys())}")

## 7. Full pipeline summary

Here is the complete pipeline from raw identifiers to a unified
perturbation space:

```
Gene symbols ──► pp.build_embedding_matrix(model=...)
                        │  (esm2_650M, enformer, esmc_600M, ...)
                        ▼
                  AnnData (.obsm["X_<model>"])
                        │
                  reduce_embeddings(PCA)
                        │
                        ▼
Drug names/SMILES ──► pp.build_molecule_embedding_matrix(
   (list or               adata=..., column="drug_name",
    AnnData.obs)          model=...)
                        │  Transformer: chemberta2MTR, molformer_base, ...
                        │  GNN:         minimol, mhg_gnn, mole
                        │  RDKit:       morgan_fp, maccs_fp, ...
                        ▼
                  AnnData (.obsm["X_<model>"])
                        │
                  reduce_embeddings(PCA)
                        │
                        ▼
              combine_perturbation_spaces()
                        │
                        ▼
              Unified AnnData with .obsm["X_combined"]
```

## Summary

| Step | Function | Notes |
|---|---|---|
| Gene embedding matrix | `pp.build_embedding_matrix(ids, model)` | Any gene/protein/DNA model |
| Drug embedding matrix (from list) | `pp.build_molecule_embedding_matrix(identifiers=[...], model=...)` | Transformer, GNN, or RDKit models |
| Drug embedding matrix (from AnnData) | `pp.build_molecule_embedding_matrix(adata=adata, column="drug_name", model=...)` | Same model support |
| Filter failures | `pp.filter_failed_embeddings(adata)` | |
| PCA reduction | `reduce_embeddings(adata, obsm_key, n_components)` | With optional StandardScaler |
| Combine spaces | `pp.combine_perturbation_spaces(adata1, adata2)` | |
| Save / load | `adata.write_h5ad()` / `ad.read_h5ad()` | |